In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yaml
import os
import sys
# add the project root to the path so the src package can be imported
sys.path.append(os.path.abspath('..'))
from src.inference_pipeline import run_pipeline,load_recommended_models
from src.vis import (
    visualize_quadrant_stage, visualize_teeth_stage,
    visualize_healthy_unhealthy_stage, visualize_disease_stage, visualize_final_result
)
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np

with open('../configs/trained_models.yaml', 'r') as f:
    all_trained_models = yaml.safe_load(f)
    final_models = all_trained_models['final_recommended_models']

In [6]:
models = load_recommended_models(
    final_models,
    device='cuda'
)

clear_output(wait=True)

class_names = {
    'teeth': [str(i) for i in range(8)],
    'healthy_unhealthy': ['Disease Found', 'Healthy'],
    'disease': ['Impacted', 'Caries', 'Periapical'],
    'caries_severity': ['Caries', 'Deep Caries'],
}

testset_path = r'..\Data\Raw\DENTEX CHALLENGE 2023\Test_Data\quadrant_enumeration_disease\xrays'
random_testset_image = np.random.choice(os.listdir(testset_path))
random_image_path = os.path.join(testset_path,random_testset_image)

result, warnings = run_pipeline(
    image_path=random_image_path,
    models=models,
    class_names=class_names,
    device='cuda'
)


print('Warnings:', warnings)
print('Diseased teeth found:', len(result['diseased_teeth']))
for t in result['diseased_teeth']:
    probs = {
        cls: f"{prob * 100:.2f}%"
        for cls, prob in t.get('disease_probs', {}).items()
    }

    print(t['quad_key'],t.get('class_name'),t.get('disease'),probs,t.get('caries_severity'),t.get('caries_severity_probs'))

100%|██████████| 4/4 [00:00<00:00, 13.15it/s]


Warnings: ['needs_manual_review: tooth tooth 7 box is 1.9x the median width, may span two teeth | tooth 6 box is 1.8x the median width, may span two teeth | tooth 5 box is 1.8x the median width, may span two teeth (confidence: nan)', 'needs_manual_review: tooth tooth 7 box is 2.1x the median width, may span two teeth | tooth 5 box is 1.7x the median width, may span two teeth | tooth 6 box is 1.9x the median width, may span two teeth (confidence: nan)']
Diseased teeth found: 7
train_48_LowerLeft 7 Caries {'Impacted': '15.11%', 'Caries': '64.49%', 'Periapical': '20.40%'} Deep Caries {'Caries': 31.87, 'Deep Caries': 68.13}
train_48_LowerLeft 6 Caries {'Impacted': '7.20%', 'Caries': '83.56%', 'Periapical': '9.24%'} Caries {'Caries': 61.42, 'Deep Caries': 38.58}
train_48_LowerLeft 5 Caries {'Impacted': '2.26%', 'Caries': '78.73%', 'Periapical': '19.01%'} Caries {'Caries': 70.39, 'Deep Caries': 29.61}
train_48_LowerRight 7 Caries {'Impacted': '4.40%', 'Caries': '64.26%', 'Periapical': '31.33